In [4]:
!pip install -q ultralytics || echo "installation failed"
!pip install -q kagglehub || echo "installation failed"


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
from pathlib import Path
import kagglehub
import shutil
import cv2
import numpy as np
from ultralytics import YOLO
import yaml

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\z004vpub\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
# Download latest version
path = kagglehub.dataset_download("hemanthganeshvilluri/cats-dogs-yolo")

print("Path to dataset files:", path)

# Define destination path
dest_path = Path("dataset/dog_and_cat")
dest_path.mkdir(parents=True, exist_ok=True)

# Copy dataset files to destination
for item in Path(path).iterdir():
    if item.is_file():
        shutil.copy2(item, dest_path)
    elif item.is_dir():
        shutil.copytree(item, dest_path / item.name, dirs_exist_ok=True)

print(f"Dataset copied to: {dest_path}")

Path to dataset files: C:\Users\z004vpub\.cache\kagglehub\datasets\hemanthganeshvilluri\cats-dogs-yolo\versions\1
Dataset copied to: dataset\dog_and_cat


In [7]:
train_path = dest_path / "train" / "images"
test_path = dest_path / "test" / "images"
valid_path = dest_path / "valid" / "images"

train_length = len(list(train_path.glob("*"))) if train_path.exists() else 0
test_length = len(list(test_path.glob("*"))) if test_path.exists() else 0
valid_length = len(list(valid_path.glob("*"))) if valid_path.exists() else 0

print(f"Train folder length: {train_length}")
print(f"Test folder length: {test_length}")
print(f"Valid folder length: {valid_length}")

Train folder length: 960
Test folder length: 52
Valid folder length: 39


In [9]:
# Initialize lists for images and labels
x_train = []
y_train = []

# Get all image files from train folder
image_files = sorted(list(train_path.glob("*.jpg")) + list(train_path.glob("*.png")))

# Path to labels folder
train_labels_path = dest_path / "train" / "labels"

for img_file in image_files:
    # Load image
    img = cv2.imread(str(img_file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    x_train.append(img)
    
    # Load corresponding label file
    label_file = train_labels_path / f"{img_file.stem}.txt"
    if label_file.exists():
        with open(label_file, 'r') as f:
            label_data = f.read().strip()
            # Extract class label (first value in YOLO format)
            if label_data:
                class_id = int(label_data.split()[0])
                y_train.append(class_id)
            else:
                y_train.append(-1)  # No label found
    else:
        y_train.append(-1)  # No label file

# Convert to numpy arrays
x_train = np.array(x_train)
y_train = np.array(y_train)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Unique labels: {np.unique(y_train)}")

x_train shape: (960, 640, 640, 3)
y_train shape: (960,)
Unique labels: [0 1]


In [11]:
# Create data.yaml file for YOLOv8
data_config = {
    'path': str(dest_path.absolute()),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,  # number of classes
    'names': ['cat', 'dog']  # class names
}

# Save data.yaml
yaml_path = dest_path / 'csb_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

# Initialize YOLOv8 model
model = YOLO('yolov8n.pt')  # using nano model for faster training

# Train the model
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    name='dog_cat_detector'
)

Ultralytics 8.4.21  Python-3.12.6 torch-2.5.1+cpu CPU (11th Gen Intel Core i7-11850H @ 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\dog_and_cat\csb_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dog_cat_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True